# Version 26 — Robust / stochastic MPC for unseen buildings

**Hypothesis:** An uncertainty-aware, finite-horizon controller will generalize better to unseen buildings than nominal MPC by accounting for thermal-model errors, not just their best-fit estimates.

## How it differs from Version 12

1. **Learn two forms of uncertainty per zone:** bounded RLS covariance for uncertain response parameters; an exponentially weighted variance of the **previous issued action's** thermal prediction error for unmodeled disturbances. Discontinuous sensors clear the pending transition, preventing a subsequent multi-step response from polluting one-step identification.
2. **Forecast disturbance envelopes:** over 20 and 50 minutes, random prediction errors grow proportional to the square root of time; persistent parameter error grows linearly with horizon. Both are capped to keep the algorithm stable.
3. **Evaluate three scenarios:** nominal, warm-tail and cold-tail temperatures for each feasible AHU mode and VAV profile. The risk-weighted objective combines the nominal result with the scenario mean and worst tail. This is scenario-based **approximate robust / stochastic MPC**, not a full plant simulator or a statistically calibrated chance-constrained optimizer.
4. **Add a directional 20-minute risk screen:** a hot/cold tail excursion beyond 1.5 K prohibits switching the AHU off. For an unambiguous one-sided risk it also rejects an actively counterproductive heat-only/cool-only AHU mode. Missing outdoor-temperature readings enlarge the forecast envelope. The original severe-state safeguards, VAV actuator constraints, economizer and shared hot-water dispatch remain.
5. **Keep computation bounded:** four modes; no more than three VAV profiles per running mode; two forecast checkpoints; three temperature scenarios per checkpoint; no SciPy, external model, or offline training.

## Controlled experiment

- Set `TransferPolicy.USE_UNCERTAINTY = False` to ablate uncertainty and recover V12's nominal comfort scoring and current-state guardrail.
- Set `TransferPolicy.RISK_WEIGHT = 0.0` to isolate nominal comfort scoring while leaving the optional off guard intact.
- Evaluate **V7 (public score 0.03947), V12, V26 nominal ablation, and V26 robust** on the **same unseen building IDs, weather realizations, and 8,640-step rollouts**.
- Compare official normalized score (lower is better), comfort exceedance duration and maximum excursions, electricity, gas, carbon, AHU on-time, and runtime. Report mean and worst-building results, not just the aggregate average. Tune only on training buildings.

## Limitations

RLS covariance is a shape/direction proxy, not a calibrated posterior. Innovation variance is an EWMA of observed one-step model errors and may misrepresent nonstationary regimes. The scenarios provide a practical risk buffer, **not** coverage guarantees or proof of robust feasibility. Action-conditioned forecasts assume approximately constant commands over short horizons. Energy terms are the existing V12 proxies rather than true kWh. The competition simulator has **not** been run and no performance gain is claimed.

The first code cell defines `TransferPolicy`; the final code cell runs the competition gateway. Submit this notebook directly in Kaggle.


In [ ]:
"""Version 26: uncertainty-aware, bounded scenario MPC for unseen buildings.

Adapts each zone's action-conditioned linear model with online RLS.
Uses the RLS covariance and an EWMA one-step innovation variance as
conservative, HEURISTIC forecast uncertainty; evaluates nominal and
two symmetric disturbance scenarios at two receding-horizon checkpoints.
A tail-risk objective and a conservative off-mode guard handle mismatch.

No distribution-free confidence guarantee or official performance claim.
Only NumPy and the Kaggle evaluation gateway are required.
"""
import re
from typing import NamedTuple

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class OperatingPoint(NamedTuple):
    """One physical AHU/plant operating point in Kelvin and pascals."""

    mode: str
    running: bool
    supply_k: float
    fan_pa: float
    water_k: float
    water_on: bool


class ZoneResponseModel:
    """Fit one zone's five-minute Kelvin change using bounded online RLS.

    Regression: ΔT = b0 * ((Tout-T)/20) + b1 * cooling + b2 * reheat + b3.
    The features encode *issued actions*, so the model can compare the
    consequences of candidate commands, unlike a passive trend estimator.
    Coefficient signs are constrained by physical direction; the priors are
    used from step one and the covariance limits the size of updates.
    """

    MAX_COVARIANCE_DIAGONAL = 8.0
    INITIAL_COVARIANCE = (0.45, 0.55, 0.55, 0.12)

    # Residual noise and model-parameter uncertainty are separate.
    # These are practical uncertainty surrogates, not calibrated intervals.
    INNOVATION_SMOOTHING = 0.08
    MIN_INNOVATION_VARIANCE = 0.0036  # (0.06 K / step)^2
    MAX_INNOVATION_VARIANCE = 0.16    # (0.40 K / step)^2
    PARAMETER_SIGMA_SCALE = 0.045
    COLD_START_SIGMA = 0.045
    MAX_FORECAST_SIGMA = 2.5

    def __init__(self):
        self.weights = np.array([0.035, -0.16, 0.20, 0.0], dtype=np.float64)
        self.covariance = np.diag(self.INITIAL_COVARIANCE)
        self.last_temperature = None
        self.last_features = None
        self.samples = 0
        self.innovation_variance = 0.0225  # 0.15 K initial one-step residual

    @staticmethod
    def features(temp, outdoor, run, supply, damper, valve, water_on, water_temp):
        """Produce action-dependent ambient/cooling/reheat/bias features.

        Temperatures use Kelvin; damper/valve are fractions, not commands.
        """
        outdoor = outdoor if np.isfinite(outdoor) else temp
        ambient = np.clip((outdoor - temp) / 20.0, -3.0, 3.0)
        # AHU cooling coil cannot heat. Natural outdoor mixing can cool air
        # further than the supply setpoint even with the chiller inactive.
        mixed_air = 0.70 * temp + 0.30 * outdoor
        delivered = min(supply, mixed_air)
        cooling = run * damper * np.clip((temp - delivered) / 12.0, 0.0, 2.0)
        reheat = (run * damper * valve * water_on *
                  np.clip((water_temp - temp) / 55.0, 0.0, 2.0))
        return np.array([ambient, cooling, reheat, 1.0], dtype=np.float64)

    def observe(self, current_temperature):
        """Update RLS coefficients from the previous issued action's outcome.

        Forgetting factor 0.995; clip residual and physically directional
        coefficients; discard implausible >1.5K five-minute jumps.
        """
        if self.last_temperature is None or self.last_features is None:
            return
        delta = current_temperature - self.last_temperature
        if not np.isfinite(delta) or abs(delta) > 1.5:
            # The previous observation/action pair cannot be used again on
            # the next step: that would fit a multi-step response as one step.
            self.clear_pending()
            return
        x = self.last_features
        px = self.covariance @ x
        denominator = 0.995 + x @ px
        if not np.isfinite(denominator) or denominator <= 1e-10:
            # Recover from ill-conditioned identification rather than letting
            # one numerical failure contaminate every later action.
            self.covariance = np.diag(self.INITIAL_COVARIANCE)
            self.clear_pending()
            return

        gain = px / denominator
        predicted = float(self.weights @ x)
        innovation = delta - predicted

        # Learn disturbance scale from the action that was ACTUALLY issued.
        # Cap outliers so occasional sensor noise does not disable the plant.
        squared_residual = min(0.64, innovation * innovation)
        self.innovation_variance = float(np.clip(
            (1.0 - self.INNOVATION_SMOOTHING) * self.innovation_variance
            + self.INNOVATION_SMOOTHING * squared_residual,
            self.MIN_INNOVATION_VARIANCE,
            self.MAX_INNOVATION_VARIANCE,
        ))
        self.weights += gain * np.clip(innovation, -0.8, 0.8)
        self.weights = np.clip(
            self.weights,
            [0.002, -0.65, 0.004, -0.12],
            [0.20, -0.004, 0.70, 0.12],
        )

        self.covariance = (self.covariance - np.outer(gain, px)) / 0.995
        self.covariance = 0.5 * (self.covariance + self.covariance.T)

        # RLS forgetting otherwise inflates unexcited covariance directions
        # by 1/0.995 at every step; this episode lasts 8,640 steps.
        diagonal = np.diag(self.covariance)
        if (not np.all(np.isfinite(self.covariance)) or
                np.any(diagonal <= 0.0)):
            self.covariance = np.diag(self.INITIAL_COVARIANCE)
        else:
            largest = float(np.max(diagonal))
            if largest > self.MAX_COVARIANCE_DIAGONAL:
                self.covariance *= self.MAX_COVARIANCE_DIAGONAL / largest
        self.samples += 1

    def predict_delta(self, features):
        return float(np.clip(self.weights @ features, -0.50, 0.50))

    def forecast_sigma(self, features, steps):
        """Forecast temperature spread in K at a future five-minute step.

        Random innovations accumulate as sqrt(horizon), while uncertain
        thermal-response coefficients can bias every future step and thus
        accumulate linearly. The coefficient scale is heuristic: RLS P is
        not a calibrated posterior covariance with these bounded priors.
        """
        features = np.asarray(features, dtype=np.float64)
        if features.shape != (4,) or not np.all(np.isfinite(features)):
            return self.MAX_FORECAST_SIGMA
        horizon = max(1, int(steps))
        parameter_variance = float(features @ self.covariance @ features)
        if not np.isfinite(parameter_variance):
            return self.MAX_FORECAST_SIGMA
        parameter_variance = max(0.0, min(16.0, parameter_variance))
        parameter_sigma = (
            self.PARAMETER_SIGMA_SCALE * np.sqrt(parameter_variance)
            + self.COLD_START_SIGMA / np.sqrt(self.samples + 1.0)
        )
        spread_squared = (
            horizon * self.innovation_variance
            + (horizon * parameter_sigma) ** 2
        )
        return float(min(self.MAX_FORECAST_SIGMA, np.sqrt(spread_squared)))

    def remember(self, temperature, features):
        self.last_temperature = float(temperature)
        self.last_features = features.copy()

    def clear_pending(self):
        self.last_temperature = None
        self.last_features = None


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Risk-aware finite-horizon MPC with online identification and safe HVAC.

    Each AHU searches four physical modes and up to three VAV profiles per
    zone. The objective blends nominal, expected three-scenario and worst
    tail discomfort. The physical fan/cooling/reheat energy proxy remains
    identical to V12 for a controlled experiment.
    """

    # Four plant modes and at most three VAV profiles per running mode:
    # 1 + 3 + 3 + 3 = 10 profiles/zone, versus 49 in Version 6.
    # Two checkpoints versus V6's three: 20 and 50 minutes.
    LEARN_DYNAMICS = True
    OPTIMIZE_ENERGY = True
    PLAN_STEPS = (4, 10)
    DISCOMFORT_WEIGHT = 1.0
    ENERGY_WEIGHT = 1.0

    # ABLATION: setting USE_UNCERTAINTY=False reproduces V12's nominal
    # comfort objective and original severe-state guardrail.
    USE_UNCERTAINTY = True
    RISK_WEIGHT = 0.65      # Blend nominal vs scenario-risk loss
    SCENARIO_SPREAD = 1.28  # Heuristic scale; not a confidence guarantee
    SCENARIO_MEAN_WEIGHT = 0.55
    MAX_OFF_FORECAST_VIOLATION_K = 1.5
    MISSING_WEATHER_DRIFT_K_PER_STEP = 0.06

    # These are scenario weights, not calibrated probability estimates.
    SCENARIO_WEIGHTS = (0.60, 0.20, 0.20)  # nominal, warm, cold

    # "mixed" retains both cooling and local reheat in a single AHU.
    OPERATING_POINTS = (
        OperatingPoint('off', False, 305., 200., 310., False),
        OperatingPoint('cool', True, 289., 760., 310., False),
        OperatingPoint('heat', True, 305., 760., 348., True),
        OperatingPoint('mixed', True, 291., 800., 340., True),
    )

    def __init__(self, building_meta_data, action_spec):
        """Map variable-sized building sensors and actuator locations from metadata."""
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.continuous_count = int(m['n_continuous_action'])
        self.discrete_count = int(m['n_discrete_action'])
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.ahu_supply_actions = [int(x) for x in m['temp_indices']]
        self.ahu_pressure_actions = [int(x) for x in m['pressure_indices']]
        self.ahu_recirc_actions = [int(x) for x in m['recirculation_indices']]
        self.water_actions = [int(x) for x in m['water_indices']]
        self.water_temp_action = self.water_actions[0] if self.water_actions else None
        n_ahu = len(self.ahu_supply_actions)
        self.water_pressure_action = (self.ahu_pressure_actions[n_ahu]
                                      if len(self.ahu_pressure_actions) > n_ahu else None)
        self.water_run_action = self.discrete_count - 1
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        normalizers = m.get('observation_normalization_metadata', {})
        temp_normalizer = normalizers.get('zone_air_temperature_sensor', {})
        self.temp_mean = float(temp_normalizer.get('sample_mean', 0.0))
        self.temp_std = float(temp_normalizer.get('sample_std', 1.0))
        if not np.isfinite(self.temp_std) or self.temp_std <= 0.0:
            self.temp_std = 1.0

        # Outdoor-air observations are NORMALIZED too. Without valid
        # normalization metadata, treat outdoor weather as unknown rather than
        # interpreting a z-score (e.g. -2.0) as a temperature in Kelvin.
        outdoor_normalizer = normalizers.get('outside_air_temperature_sensor', {})
        self.outdoor_mean = float(outdoor_normalizer.get('sample_mean', np.nan))
        self.outdoor_std = float(outdoor_normalizer.get('sample_std', np.nan))
        if (not np.isfinite(self.outdoor_mean) or
                not np.isfinite(self.outdoor_std) or
                self.outdoor_std <= 0.0):
            self.outdoor_mean = np.nan
            self.outdoor_std = np.nan

        def numeric_order(value):
            match = re.search(r'\d+', str(value))
            return (int(match.group()) if match else -1, str(value))

        self.ahus = []
        ahu_ids = sorted(m['ahu_zone_assignments'], key=numeric_order)[:n_ahu]
        for number, ahu_id in enumerate(ahu_ids):
            zones = []
            for room in m['ahu_zone_assignments'][ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                sensor = self.fields.get(vav + '_zone_air_temperature_sensor')
                if sensor is None:
                    continue
                actions = m['zone_action_indices'].get(vav, {})
                zones.append({
                    'name': vav,
                    'sensor': sensor,
                    'heat_setpoint': self.fields.get(
                        vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'cool_setpoint': self.fields.get(
                        vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper_action': actions.get(
                        'supply_air_damper_percentage_command'),
                    'valve_action': actions.get(
                        'reheat_valve_percentage_command'),
                    'model': ZoneResponseModel(),
                })
            self.ahus.append({
                'zones': zones, 'run_action': number,
                'supply_action': self.ahu_supply_actions[number],
                'pressure_action': (
                    self.ahu_pressure_actions[number]
                    if number < len(self.ahu_pressure_actions) else None
                ),
                'recirc_action': (
                    self.ahu_recirc_actions[number]
                    if number < len(self.ahu_recirc_actions) else None
                ),
                'outside_sensor': self.fields.get(
                    'ahs_' + str(ahu_id) + '_outside_air_temperature_sensor'),
            })
        self.reset()

    def reset(self):
        """Isolate episodes, including online model state and pending actions."""
        for ahu in self.ahus:
            for zone in ahu['zones']:
                zone['model'] = ZoneResponseModel()

    @staticmethod
    def _command(fraction):
        """Map physical actuator fraction [0, 1] to normalized [-1, 1]."""
        return float(np.clip(2.0 * fraction - 1.0, -1.0, 1.0))

    @staticmethod
    def _supply_command(kelvin):
        """Map 285..305 Kelvin supply-air setpoint into [-1, 1]."""
        return float(np.clip((kelvin - 285.0) / 10.0 - 1.0, -1.0, 1.0))

    def _fallback_band(self, obs):
        """Use weekday/hour only if current zone setpoint sensors are invalid."""
        def cyclic(sine, cosine, period):
            a, b = self.fields.get(sine), self.fields.get(cosine)
            if a is None or b is None:
                return None
            if not np.isfinite(obs[a]) or not np.isfinite(obs[b]):
                return None
            return (np.arctan2(obs[a], obs[b]) %
                    (2.0 * np.pi)) * period / (2.0 * np.pi)
        hour = cyclic('hod_sin_000', 'hod_cos_000', 24)
        day = cyclic('dow_sin_000', 'dow_cos_000', 7)
        occupied = (hour is not None and day is not None and day < 5 and
                    self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zones(self, obs, ahu, fallback):
        """Denormalize zone temperatures, validate Kelvin setpoints, and learn.

        Observe the temperature response to the PREVIOUS issued action before
        selecting today's candidate. Invalid sensors clear pending transitions.
        """
        result = []
        outdoor_idx = ahu['outside_sensor']
        outdoor = np.nan
        if outdoor_idx is not None and np.isfinite(self.outdoor_std):
            normalized = float(obs[outdoor_idx])
            if np.isfinite(normalized):
                decoded = self.outdoor_mean + normalized * self.outdoor_std
                if 180.0 <= decoded <= 350.0:
                    outdoor = decoded
        for zone in ahu['zones']:
            temp = float(obs[zone['sensor']]) * self.temp_std + self.temp_mean
            if not np.isfinite(temp) or not 240.0 <= temp <= 340.0:
                zone['model'].clear_pending()
                continue
            li, ui = zone['heat_setpoint'], zone['cool_setpoint']
            low = float(obs[li]) if li is not None else np.nan
            high = float(obs[ui]) if ui is not None else np.nan
            if not (np.isfinite(low) and np.isfinite(high) and
                    245.0 <= low < high <= 335.0 and high-low <= 35.0):
                low, high = fallback
            if self.LEARN_DYNAMICS:
                zone['model'].observe(temp)
            result.append({'zone': zone, 'temp': temp, 'low': low,
                           'high': high, 'outdoor': outdoor})
        return result

    @staticmethod
    def _violation(temp, low, high):
        """Comfort deviation in K, with convex penalty beyond the 5K knee."""
        outside = max(low - temp, temp - high, 0.0)
        return outside / 5.0 if outside <= 5.0 else 1.0 + ((outside - 5.0) / 5.0)**2

    @staticmethod
    def _profiles(config, zone):
        """Small, commandable VAV grid for one AHU mode.

        With no reheat, search minimum ventilation, medium, and full airflow.
        With reheat, search minimum ventilation, full cooling, and full heat.
        Missing actuator mappings cannot be optimized as if controllable.
        """
        if not config.running:
            return ((0.0, 0.0),)

        can_heat = config.water_on and zone['valve_action'] is not None
        if zone['damper_action'] is None:
            return ((0.1, 0.0), (0.1, 1.0)) if can_heat else ((0.1, 0.0),)
        if can_heat:
            return ((0.1, 0.0), (1.0, 0.0), (1.0, 1.0))
        return ((0.1, 0.0), (0.55, 0.0), (1.0, 0.0))

    def _scenario_radius(self, state, features, steps):
        """Return a conservative symmetric forecast excursion in Kelvin.

        The zone model estimates innovation and parameter uncertainty. When
        outdoor-air temperature is missing, add an independent drift allowance
        instead of assuming the weather-driven prediction is equally reliable.
        """
        sigma = state['zone']['model'].forecast_sigma(features, steps)
        if not np.isfinite(state['outdoor']):
            missing_weather_sigma = (
                self.MISSING_WEATHER_DRIFT_K_PER_STEP * steps
            )
            sigma = min(
                state['zone']['model'].MAX_FORECAST_SIGMA,
                float(np.hypot(sigma, missing_weather_sigma)),
            )
        return self.SCENARIO_SPREAD * sigma

    def _forecast_discomfort(self, state, rate, features):
        """Score nominal, warm-tail and cold-tail comfort at each checkpoint.

        The planner applies one action at a time and replans on the next
        observation. This is a scenario approximation, not a calibrated
        stochastic optimizer or a formal chance-constrained guarantee.
        """
        total = 0.0
        risk_enabled = self.USE_UNCERTAINTY and self.RISK_WEIGHT > 0.0
        nominal_weight, warm_weight, cold_weight = self.SCENARIO_WEIGHTS

        for steps in self.PLAN_STEPS:
            expected_temperature = state['temp'] + rate * steps
            nominal = self._violation(
                expected_temperature, state['low'], state['high']
            )

            if not risk_enabled:
                total += nominal
                continue

            radius = self._scenario_radius(state, features, steps)
            warm = self._violation(
                expected_temperature + radius, state['low'], state['high']
            )
            cold = self._violation(
                expected_temperature - radius, state['low'], state['high']
            )

            scenario_mean = (
                nominal_weight * nominal
                + warm_weight * warm
                + cold_weight * cold
            )
            scenario_tail = max(warm, cold)
            scenario_risk = (
                self.SCENARIO_MEAN_WEIGHT * scenario_mean
                + (1.0 - self.SCENARIO_MEAN_WEIGHT) * scenario_tail
            )
            total += (
                (1.0 - self.RISK_WEIGHT) * nominal
                + self.RISK_WEIGHT * scenario_risk
            )

        return total / len(self.PLAN_STEPS)

    def _off_tail_violations(self, state):
        """Predict hot and cold comfort-band overshoot with the AHU off.

        Returns separate nonnegative exceedances, allowing one-sided risk
        to prohibit an actively counterproductive HVAC operating mode.
        """
        model = state['zone']['model']
        features = model.features(
            state['temp'], state['outdoor'], 0.0,
            305.0, 0.0, 0.0, 0.0, 310.0,
        )
        steps = self.PLAN_STEPS[0]
        expected_temperature = (
            state['temp'] + model.predict_delta(features) * steps
        )
        radius = self._scenario_radius(state, features, steps)
        hot = max(0.0, expected_temperature + radius - state['high'])
        cold = max(0.0, state['low'] - (expected_temperature - radius))
        return hot, cold

    def _off_forecast_violation(self, state):
        """Maximum of hot/cold off-mode exceedances, in Kelvin."""
        return max(self._off_tail_violations(state))

    def _best_vav(self, state, config):
        """Select the lowest-risk, commandable damper/reheat profile.

        Reuse V12's action-conditioned linear dynamics and energy proxy.
        Only uncertainty-based comfort scoring changes when enabled.
        """
        zone, temperature = state['zone'], state['temp']
        model = zone['model']
        weights = model.weights
        outdoor = state['outdoor']
        if not np.isfinite(outdoor):
            outdoor = temperature

        ambient = max(-3.0, min(3.0, (outdoor - temperature) / 20.0))
        base_rate = float(weights[0]) * ambient + float(weights[3])
        cooling_feature = 0.0
        heating_feature = 0.0
        if config.running:
            delivered = min(
                config.supply_k, 0.70 * temperature + 0.30 * outdoor)
            cooling_feature = max(
                0.0, min(2.0, (temperature - delivered) / 12.0))
            if config.water_on:
                heating_feature = max(
                    0.0, min(2.0, (config.water_k - temperature) / 55.0))

        best = (float('inf'), 0.0, 0.0)
        for damper, valve in self._profiles(config, zone):
            cooling_load = damper * cooling_feature
            reheat_load = damper * valve * heating_feature
            predicted_rate = max(-0.50, min(
                0.50,
                base_rate
                + float(weights[1]) * cooling_load
                + float(weights[2]) * reheat_load,
            ))
            features = (ambient, cooling_load, reheat_load, 1.0)
            comfort = self._forecast_discomfort(
                state, predicted_rate, features
            )
            score = self.DISCOMFORT_WEIGHT * comfort
            if self.OPTIMIZE_ENERGY:
                score += self.ENERGY_WEIGHT * (
                    (0.012 * damper if config.running else 0.0) +
                    (0.022 * damper * valve if config.water_on else 0.0))
            if score < best[0]:
                best = (score, damper, valve)
        return best

    @staticmethod
    def _plant_cost(config):
        """Estimate relative fan, cooling and boiler cost (not true kWh)."""
        if not config.running:
            return 0.0
        fan = 0.035 + 0.035 * (config.fan_pa / 2000.0)**1.5
        cooling = 0.025 * max(0.0, 299.0 - config.supply_k) / 14.0
        heating = (0.025 * max(0.0, config.water_k - 310.0) / 40.0
                   if config.water_on else 0.0)
        return fan + cooling + heating

    @staticmethod
    def _candidate_allowed(config, worst_hot, worst_cold):
        """Keep a corrective mode available for every severe demand.

        Off is forbidden at 2 K violation; one-sided modes are forbidden at
        4 K in the opposite direction. Mixed always remains feasible when
        different zones need simultaneous heating and cooling.
        """
        mode = config.mode
        if mode == 'off' and max(worst_hot, worst_cold) >= 2.0:
            return False
        if worst_hot >= 4.0 and mode == 'heat':
            return False
        if worst_cold >= 4.0 and mode == 'cool':
            return False
        return True

    def _select_operating_point(self, zones):
        """Select an AHU mode plus per-zone VAV profiles by short search.

        Comfort and energy costs are nonnegative, so a partly scored
        candidate can be abandoned as soon as its lower bound exceeds the
        best complete candidate. This never changes the winning result
        within Version 12's reduced action space.
        """
        worst_hot = max((z['temp'] - z['high'] for z in zones), default=0.0)
        worst_cold = max((z['low'] - z['temp'] for z in zones), default=0.0)
        best_cost, best_config, best_vavs = float('inf'), None, None
        zone_count = len(zones)

        # Off-mode scenario tails only impose additional constraints when
        # uncertainty-aware planning is on. Keep hot/cold exceedance separate:
        # an active but wrong-direction mode is not a safety correction.
        hot_risk, cold_risk = 0.0, 0.0
        if self.USE_UNCERTAINTY:
            for state in zones:
                hot, cold = self._off_tail_violations(state)
                hot_risk = max(hot_risk, hot)
                cold_risk = max(cold_risk, cold)

        hot_needs_correction = (
            hot_risk >= self.MAX_OFF_FORECAST_VIOLATION_K
        )
        cold_needs_correction = (
            cold_risk >= self.MAX_OFF_FORECAST_VIOLATION_K
        )

        for config in self.OPERATING_POINTS:
            if not self._candidate_allowed(config, worst_hot, worst_cold):
                continue
            if config.mode == 'off' and (
                hot_needs_correction or cold_needs_correction
            ):
                continue
            if hot_needs_correction and not cold_needs_correction:
                if config.mode == 'heat':
                    continue
            if cold_needs_correction and not hot_needs_correction:
                if config.mode == 'cool':
                    continue
            cost = (self.ENERGY_WEIGHT * self._plant_cost(config)
                    if self.OPTIMIZE_ENERGY else 0.0)
            choices = []
            for state in zones:
                choice = self._best_vav(state, config)
                choices.append(choice)
                cost += choice[0] / zone_count
                if cost >= best_cost:
                    break
            if len(choices) == zone_count and cost < best_cost:
                best_cost, best_config, best_vavs = cost, config, choices
        return best_config, best_vavs

    def _economizer(self, outdoor, zone_temperature, supply, pressure):
        """Choose recirculation when free cooling saves more than fan work.

        Returned +1 requests maximum recirculation, capped by the simulator.
        """
        if not np.isfinite(outdoor) or outdoor >= zone_temperature:
            return 1.0
        if zone_temperature - outdoor <= pressure * 0.00322:
            return 1.0
        fraction_outside = np.clip((zone_temperature - supply) /
                                   max(1e-6, zone_temperature-outdoor), 0.3, 1.0)
        return self._command(1.0 - fraction_outside)

    def _initial_actions(self):
        """Allocate safe idle commands, with correct array sizes and types.

        A normalized continuous zero is the MIDPOINT of a physical range,
        not OFF. +1 requests warmest AHU supply and maximum recirculation.
        """
        continuous = np.full(self.continuous_count, -1.0, dtype=np.float32)
        discrete = np.zeros(self.discrete_count, dtype=np.int32)
        for action_index in self.ahu_supply_actions + self.ahu_recirc_actions:
            continuous[action_index] = 1.0
        return continuous, discrete

    def _apply_ahu_action(self, ahu, states, config, continuous, discrete):
        """Command one AHU's run flag, cooling supply, fan and economizer.

        AHUs cannot heat: thermal reheat occurs downstream in the VAV boxes.
        """
        if config.running:
            discrete[ahu['run_action']] = 1
            continuous[ahu['supply_action']] = self._supply_command(
                config.supply_k)
            if ahu['pressure_action'] is not None:
                continuous[ahu['pressure_action']] = self._command(
                    config.fan_pa / 2000.0)
            if ahu['recirc_action'] is not None and config.supply_k < 300.0:
                outside = states[0]['outdoor']
                mean_zone = float(np.mean([s['temp'] for s in states]))
                continuous[ahu['recirc_action']] = self._economizer(
                    outside, mean_zone, config.supply_k, config.fan_pa)

    def _apply_vav_actions(self, states, config, choices, continuous,
                           heat_demands, pending):
        """Issue achievable zone commands and collect model update inputs.

        We defer saving the model's features until the shared boiler command
        is known: different AHUs can request different water temperatures.
        """
        for state, (_, damper, valve) in zip(states, choices):
            zone = state['zone']
            damper_index = zone['damper_action']
            valve_index = zone['valve_action']
            if valve_index is None:
                valve = 0.0
            if damper_index is None:
                damper = 0.10 if config.running else 0.0

            if damper_index is not None:
                continuous[damper_index] = self._command(damper)
            if valve_index is not None:
                continuous[valve_index] = self._command(valve)
            if config.running and config.water_on and valve > 0.0:
                heat_demands.append((config.water_k, config.fan_pa,
                                     damper * valve))

            pending.append((state, config, damper, valve))

    def _apply_shared_hot_water(self, heat_demands, continuous, discrete):
        """Combine all AHUs' valid reheat requests into one water command.

        Returns the common commanded water temperature, or None if unused.
        """
        if not heat_demands:
            return None

        discrete[self.water_run_action] = 1
        hot_water = max(demand[0] for demand in heat_demands)
        loading = float(np.mean([demand[2] for demand in heat_demands]))
        if self.water_temp_action is not None:
            continuous[self.water_temp_action] = self._command(
                (hot_water - 310.0) / 40.0)
        if self.water_pressure_action is not None:
            continuous[self.water_pressure_action] = self._command(
                max(0.35, loading))
        return hot_water

    @staticmethod
    def _remember_issued_actions(pending, common_water_k):
        """Store jointly issued setpoints for the next RLS update.

        These are commands, not necessarily the measured actuator positions:
        the simulator can impose floors and thermal response takes time.
        """
        for state, config, damper, valve in pending:
            heat_enabled = (common_water_k is not None and config.running
                            and config.water_on and valve > 0.0)
            water_k = common_water_k if heat_enabled else config.water_k
            features = state['zone']['model'].features(
                state['temp'], state['outdoor'], float(config.running),
                config.supply_k, damper, valve,
                float(heat_enabled), water_k)
            state['zone']['model'].remember(state['temp'], features)

    def get_action(self, obs):
        """One step: learn previous outcome -> plan -> issue -> remember.

        (1) Decode sensors and learn from the preceding issued commands.
        (2) Search four AHU modes and at most three VAV profiles per zone,
            scoring nominal and plausible high/low temperature scenarios.
        (3) Issue AHU/VAV commands and coordinate their shared hot-water loop.
        (4) Save the joint issued action for the next model observation.
        (5) Return normalized, typed hybrid action arrays.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._initial_actions()
        fallback = self._fallback_band(obs)
        heat_demands = []
        pending = []

        for ahu in self.ahus:
            states = self._read_zones(obs, ahu, fallback)
            if not states:
                continue
            config, choices = self._select_operating_point(states)
            self._apply_ahu_action(ahu, states, config, continuous, discrete)
            self._apply_vav_actions(
                states, config, choices, continuous, heat_demands, pending)

        common_water_k = self._apply_shared_hot_water(
            heat_demands, continuous, discrete)
        self._remember_issued_actions(pending, common_water_k)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
# Official Kaggle evaluation gateway. Keep this as the LAST code cell.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
